### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 3 - Kardiotoksyczność**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. hERG (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. VDss (Lombardo)
5. AMES Mutagenicity

Wyniki dla STL:

In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [ ]:
class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer): #dodane zabezpieczenia na Nan etc
    def __init__(self, y_column='Y', radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
            else:
                # Zamiast pomijać wiersz, wstawiamy wektor zer (zachowuje wyrównanie)
                fingerprints.append(np.zeros(self.length))

            # Pobieramy etykietę jeśli istnieje (dla dummy X wstawiamy NaN)
            labels.append(row[self.y_column] if self.y_column in df.columns else np.nan)

        return np.array(fingerprints), np.array(labels).reshape(-1, 1)

In [ ]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [ ]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [ ]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=1024, reg_tasks=reg_tasks, class_tasks=class_tasks).to(device)

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0

    # Include log_vars in the optimizer
    optimizer = optim.Adam(list(model.parameters()) + list(log_vars.values()), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none')
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [ ]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    # To zapobiega rozbieżnościom rozmiarów X i y
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

# Test 1: hERG (Wang) + Lipophilicity (AstraZeneca)

Test pierwszy sprawdzający czy lipofilowość pomoże w predykcji kardiotoksyczności (hERG). Zgodnie z hipotezą biologiczną wyższa lipofilowość sprzyja blokowaniu kanału hERG. Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ']
class_tasks = ['hERG'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_herg, test_herg = load_split_pickle('hERG')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_lipo, train_herg], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_lipo, test_herg],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

[17:27:16] WARNING: not removing hydrogen atom without neighbors
[17:27:18] WARNING: not removing hydrogen atom without neighbors
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use MorganGenerator
[17:27:19] DEPRECATION WARNING: please use M


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.0170
  Epoka  20 | Total Loss: 1.1309
  Epoka  40 | Total Loss: 0.4614
  Epoka  60 | Total Loss: 0.1849
  Epoka  80 | Total Loss: 0.0873

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8144
  MAE      : 0.5929
  R²       : 0.5511


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7328
  F1       : 0.8276
  AUROC    : 0.7630


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9959
  Epoka  20 | Total Loss: 0.5898
  Epoka  40 | Total Loss: 0.2434
  Epoka  60 | Total Loss: 0.1141
  Epoka  80 | Total Loss: 0.0520

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Uniform Average
  RMSE     : 0.8159
  MAE      : 0.6000
  R²       : 0.5494


Wyniki

# Test2: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB)

Dokładamy rozpuszczalność wodną (Solubility). Sprawdzamy czy właściwości fizykochemiczne wspólnie wzmocnią predykcję hERG.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['hERG'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_herg, test_herg = load_split_pickle('hERG')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_lipo, train_sol, train_herg], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_lipo, test_sol, test_herg],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:27:49] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.3534
  Epoka  20 | Total Loss: 1.3107
  Epoka  40 | Total Loss: 0.5534
  Epoka  60 | Total Loss: 0.2722
  Epoka  80 | Total Loss: 0.1647

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8292
  MAE      : 0.6094
  R²       : 0.5346


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2960
  MAE      : 0.9569
  R²       : 0.6905


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7710
  F1       : 0.8544
  AUROC    : 0.7525


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1482
  Epoka  20 | Total Loss: 0.4756
  Epoka  40 | Total Loss: 0.2082
  Epoka  60 | Total Loss: 0.0980
  Epoka  80 | Total Loss: 0.0557

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wy

# Test3: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo)

Pełny zestaw powiązanych biologicznie endpointów (hERG + 3 właściwości fizykochemiczne / dystrybucji). Sprawdzamy maksymalny pozytywny transfer w obrębie zestawu.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_herg, test_herg = load_split_pickle('hERG')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_lipo, train_sol, train_vdss, train_herg], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_lipo, test_sol, test_vdss, test_herg],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:28:13] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.4744
  Epoka  20 | Total Loss: 1.9376
  Epoka  40 | Total Loss: 0.9340
  Epoka  60 | Total Loss: 0.5137
  Epoka  80 | Total Loss: 0.2205

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7843
  MAE      : 0.5750
  R²       : 0.5837


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2709
  MAE      : 0.9381
  R²       : 0.7024


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 11.5937
  MAE      : 7.2617
  R²       : -1.8693


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7481
  F1       : 0.8390
  AUROC    : 0.7534


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1055
  Epoka  20 | Total Loss: 0.4053
  Epoka  40 | Total Loss:

# Test4: hERG (Wang) + VDss (Lombardo)

Sprawdzamy czy VDss (objętość dystrybucji w tkankach) pomoże predykcji kardiotoksyczności - leki lipofilowe gromadzące się w tkankach mają zwiększone ryzyko hERG.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['VDss_Lombardo']
class_tasks = ['hERG'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_herg, test_herg = load_split_pickle('hERG')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_vdss, train_herg], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_vdss, test_herg],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

[17:28:25] WARNING: not removing hydrogen atom without neighbors
[17:28:25] WARNING: not removing hydrogen atom without neighbors
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use MorganGenerator
[17:28:25] DEPRECATION WARNING: please use M


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.0417
  Epoka  20 | Total Loss: 1.1278
  Epoka  40 | Total Loss: 0.5074
  Epoka  60 | Total Loss: 0.2811
  Epoka  80 | Total Loss: 0.1208

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 6.1173
  MAE      : 3.8459
  R²       : 0.2012


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7710
  F1       : 0.8571
  AUROC    : 0.7967


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9507
  Epoka  20 | Total Loss: 0.5870
  Epoka  40 | Total Loss: 0.3346
  Epoka  60 | Total Loss: 0.1800
  Epoka  80 | Total Loss: 0.1125

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Uniform Average
  RMSE     : 6.3735
  MAE      : 3.8899
  R²       : 0.1328


Wyniki dla z

# Test5: hERG (Wang) + Solubility (AqSolDB)

Sprawdzamy czy sama rozpuszczalność wodna wnosi informację użyteczną do predykcji hERG.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Solubility_AqSolDB']
class_tasks = ['hERG'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_herg, test_herg = load_split_pickle('hERG')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_sol, train_herg], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_sol, test_herg],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:28:35] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.0897
  Epoka  20 | Total Loss: 0.6307
  Epoka  40 | Total Loss: 0.3104
  Epoka  60 | Total Loss: 0.1692
  Epoka  80 | Total Loss: 0.1085

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2716
  MAE      : 0.9428
  R²       : 0.7021


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7786
  F1       : 0.8585
  AUROC    : 0.7713


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.9214
  Epoka  20 | Total Loss: 0.3783
  Epoka  40 | Total Loss: 0.1504
  Epoka  60 | Total Loss: 0.0861
  Epoka  80 | Total Loss: 0.0547

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Uniform Average
  RMSE     : 1.2688
  MAE      : 0.9352
  R²       : 0.7033


Wy

# Test6: Solubility (AqSolDB) + VDss (Lombardo)

Sprawdzamy parę dwóch regresji bez hERG - czy pomocnicze endpointy uczą się razem lepiej niż osobno (kontrola wpływu głównego zadania).

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_sol, train_vdss], reg_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_sol, test_vdss],  reg_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:28:52] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.4365
  Epoka  20 | Total Loss: 1.0782
  Epoka  40 | Total Loss: 0.5953
  Epoka  60 | Total Loss: 0.3031
  Epoka  80 | Total Loss: 0.1472

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.2673
  MAE      : 0.9343
  R²       : 0.7041


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 6.1566
  MAE      : 3.8732
  R²       : 0.1909


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.4212
  Epoka  20 | Total Loss: 0.5332
  Epoka  40 | Total Loss: 0.3000
  Epoka  60 | Total Loss: 0.1536
  Epoka  80 | Total Loss: 0.0712

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Uniform Average
  RMSE     : 1.2775
  MAE      : 0.9466
  R²       : 0.6993

# Test7: hERG (Wang) + AMES Mutagenicity

Sprawdzamy łączenie z niepowiązanym biologicznie zadaniem (AMES - mutagenność). Dwa zadania klasyfikacyjne. Oczekujemy transferu negatywnego.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = []
class_tasks = ['hERG', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_herg, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_herg, test_ames],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:29:06] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.4175
  Epoka  20 | Total Loss: 0.2815
  Epoka  40 | Total Loss: 0.0380
  Epoka  60 | Total Loss: 0.0092
  Epoka  80 | Total Loss: 0.0066

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7634
  F1       : 0.8473
  AUROC    : 0.7618


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8191
  F1       : 0.8349
  AUROC    : 0.8908


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 0.6708
  Epoka  20 | Total Loss: 0.1126
  Epoka  40 | Total Loss: 0.0098
  Epoka  60 | Total Loss: 0.0039
  Epoka  80 | Total Loss: 0.0016

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Uniform Average
  Accuracy : 0.7786
  F1       : 0.8612
  AUROC    : 0.7868


Wyniki dla zadania (Kl

# Test8: hERG (Wang) + Lipophilicity (AstraZeneca) + Solubility (AqSolDB) + VDss (Lombardo) + AMES Mutagenicity

Pełny zestaw 5 endpointów: 4 powiązane biologicznie + 1 niepowiązany (AMES). Sprawdzamy łączny efekt - czy AMES zaszkodzi pozostałym, czy pełne pokrycie wygrywa.

In [ ]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo']
class_tasks = ['hERG', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "/content/drive/MyDrive/MLDD - ADMET/mtl_results_kardiotoksycznosc_fingerprints.txt"

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_vdss, test_vdss = load_split_pickle('VDss_Lombardo')
train_herg, test_herg = load_split_pickle('hERG')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_final([train_lipo, train_sol, train_vdss, train_herg, train_ames], reg_tasks + class_tasks, featurizer)
X_test_mtl,  y_test_dict  = prepare_mtl_data_final([test_lipo, test_sol, test_vdss, test_herg, test_ames],  reg_tasks + class_tasks, featurizer)

# Define all loss weighting schemes to test
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKÓW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)


    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadań regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        # Przygotowanie formatu pod Twoje funkcje (opakowanie w 'test_metrics')
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadań klasyfikacyjnych (jeśli istnieją)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypełnienie, jeśli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics( formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostały zapisane w: {filepath}")

Streaming output truncated to the last 5000 lines.
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:29:37] DEPRECATION WARNING: please use MorganGenerator
[17:2


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 5.0725
  Epoka  20 | Total Loss: 1.8799
  Epoka  40 | Total Loss: 1.0514
  Epoka  60 | Total Loss: 0.5436
  Epoka  80 | Total Loss: 0.3257

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8122
  MAE      : 0.5954
  R²       : 0.5536


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.3028
  MAE      : 0.9647
  R²       : 0.6873


Wyniki dla zadania (Regresja): VDss_Lombardo

  Loss Weighting: Weighted Loss Sum
  RMSE     : 10.4839
  MAE      : 7.4083
  R²       : -1.3463


Wyniki dla zadania (Klasyfikacja): hERG

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7634
  F1       : 0.8488
  AUROC    : 0.7546


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.8184
  F1       : 0.8346
  AUROC    : 0.8919


--- ST